# RETRAINING AND EXECUTION SCRIPTS

## RETRAINING SCRIPT

In [3]:
import pandas as pd
import numpy as np
import joblib
import os
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# =====================================================================
# 1. YOUR EXACT PATHS
# =====================================================================
BASE_PATH = "/Users/rober/cmapss-rul-prediction/"
# Using the .txt raw file as you have it defined
INPUT_FILE = os.path.join(BASE_PATH, "02_Data/01_Raw/train_FD001.txt")
OUTPUT_MODEL = os.path.join(BASE_PATH, "04_Models/nasa_model.pkl")

# Load data (handling the space-separated format of CMAPSS txt files)
# Typically CMAPSS raw txt doesn't have headers
raw_data = pd.read_csv(INPUT_FILE, sep='\s+', header=None)

# 2. FEATURE SELECTION (Matching the columns)
# Column names for FD001 raw
col_names = ['unit_number', 'time_in_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + [f'sensor_{i}' for i in range(1, 22)]
raw_data.columns = col_names

# The 8 sensors we selected for the app
selected_sensors = ['time_in_cycles', 'sensor_11', 'sensor_4', 'sensor_12', 
                   'sensor_7', 'sensor_15', 'sensor_21', 'sensor_20']

# --- CALCULATE RUL (Quick fix for retraining) ---
# This ensures 'y' exists and is consistent
max_cycles = raw_data.groupby('unit_number')['time_in_cycles'].transform('max')
raw_data['RUL'] = max_cycles - raw_data['time_in_cycles']

X = raw_data[selected_sensors]
y = raw_data['RUL']

# =====================================================================
# 3. STOCKOUT STRATEGY (NATIVE PIPELINE)
# =====================================================================
# Using native Sklearn pieces so Streamlit Cloud can read it 
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), selected_sensors)
    ]
)

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42))
])

# 4. TRAIN AND SAVE
full_pipeline.fit(X, y)

os.makedirs(os.path.dirname(OUTPUT_MODEL), exist_ok=True)
joblib.dump(full_pipeline, OUTPUT_MODEL)

print(f"✅ DONE! Model saved at: {OUTPUT_MODEL}")
print("This model is now a native Pipeline and will work on Streamlit Cloud.")

✅ DONE! Model saved at: /Users/rober/cmapss-rul-prediction/04_Models/nasa_model.pkl
This model is now a native Pipeline and will work on Streamlit Cloud.


## EXECUTION SCRIPT

In [4]:
import pandas as pd
import joblib
import os

# =====================================================================
# 1. ENVIRONMENT & PATHS
# =====================================================================
PROJECT_PATH = '/Users/rober/cmapss-rul-prediction'
VALIDATION_PATH = os.path.join(PROJECT_PATH, '02_Data/02_Validation/validation_FD001.csv')
MODEL_PATH = os.path.join(PROJECT_PATH, '04_Models/nasa_model.pkl')
RESULTS_PATH = os.path.join(PROJECT_PATH, '05_Results/predictions_validation_FD001.csv')

# =====================================================================
# 2. LOAD DATA & MODEL
# =====================================================================
# Load the raw validation data
# Note: The pipeline will handle the feature selection and renaming internally
df_raw = pd.read_csv(VALIDATION_PATH)

# Load the trained autonomous pipeline
# We use joblib as it is standard for scikit-learn pipelines
model_pipeline = joblib.load(MODEL_PATH)

# =====================================================================
# 3. EXECUTION (PREDICTION)
# =====================================================================
# We simply pass the raw dataframe. 
# The 'Inside' logic of the pipeline takes care of the rest.
predictions = model_pipeline.predict(df_raw)

# =====================================================================
# 4. CHECKING PREDICTIONS & SAVING RESULTS
# =====================================================================
# Combine predictions with reference columns for validation
results = pd.DataFrame({
    'unit_number': df_raw['unit_number'],
    'time_in_cycles': df_raw['time_in_cycles'],
    'predicted_RUL': predictions
})

# Sort for better readability
results = results.sort_values(by=['unit_number', 'time_in_cycles']).reset_index(drop=True)

# Display the first 40 rows as a check
print("--- Execution Check: Predicted RUL Samples ---")
display(results.head(40))

# Save the final results for Streamlit or further analysis
results.to_csv(RESULTS_PATH, index=False)
print(f"\nPredictions successfully saved to: {RESULTS_PATH}")

--- Execution Check: Predicted RUL Samples ---


,unit_number,time_in_cycles,predicted_RUL
0,1,1,200.673050
1,1,2,202.009918
2,1,3,207.285904
3,1,4,209.032181
4,1,5,202.644424
5,1,6,202.686417
6,1,7,204.486389
7,1,8,203.965347
8,1,9,200.717484
9,1,10,194.039246



Predictions successfully saved to: /Users/rober/cmapss-rul-prediction/05_Results/predictions_validation_FD001.csv
